In [20]:
import os
from openimages.download import download_dataset

In [ ]:
data_dir = 'data'
number_of_samples = 350
classes = ['Missile', 'Balloon', 'Castle']

In [22]:
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

In [23]:
if not (os.path.exists(os.path.join(data_dir, classes[0].lower())) and
        os.path.exists(os.path.join(data_dir, classes[1].lower())) and
        os.path.exists(os.path.join(data_dir, classes[2].lower()))):
    download_dataset(data_dir, classes, limit=number_of_samples)

In [24]:
import torch
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import numpy as np
from skimage import io
from skimage.transform import resize
from skimage.color import gray2rgb
import glob

In [25]:
def read_img(file_name):
    img = io.imread(file_name)
    if img.ndim == 2:
        img = gray2rgb(img)
    img = resize(img, (224, 224))
    img = torch.tensor(img)
    img = img.permute(2, 0, 1)
    return img.float()

In [26]:
class CustomDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = images_dir
        self.transforms = transforms

        self.class1_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[0].lower()))
        self.class2_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[1].lower()))
        self.class3_files = glob.glob(self.images_dir + '/{}/images/*.jpg'.format(classes[2].lower()))
        self.class1_len = len(self.class1_files)
        self.class2_len = len(self.class2_files)
        self.class3_len = len(self.class3_files)

        self.files = self.class1_files + self.class2_files + self.class3_files

        self.labels = np.zeros(len(self.files))
        self.labels[self.class1_len:self.class1_len + self.class2_len] = 1
        self.labels[self.class1_len + self.class2_len:] = 2

        self.order = [x for x in np.random.permutation(len(self.labels))]
        self.files = [self.files[x] for x in self.order]
        self.labels = [self.labels[x] for x in self.order]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        file = self.files[i]
        im = read_img(file)

        img = im.clone().detach()

        y = self.labels[i]
        return img, y

In [27]:
dataset = CustomDataset(data_dir)
train_dataset = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4, prefetch_factor=2)
iterator = iter(train_dataset)

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights)
model.eval()
pass

In [29]:
def prepare_batch(images):
    batch = []
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

    for img in images:
        batch.append((img - mean) / std)

    return torch.cat(batch, dim=0)

In [30]:
tp = [0, 0, 0]
fp = [0, 0, 0]
tn = [0, 0, 0]
fn = [0, 0, 0]

for images, labels in iterator:
    batch = prepare_batch(images)
    predictions = model(batch).softmax(1)

    for i, prediction in enumerate(predictions):
        top_idx = prediction.argmax()
        predicted_class = weights.meta['categories'][top_idx]
        true_label = int(labels[i])

        for c in range(3):
            predicted_is_c = predicted_class == classes[c].lower()
            true_is_c = true_label == c

            if true_is_c and predicted_is_c:
                tp[c] += 1
            elif not true_is_c and predicted_is_c:
                fp[c] += 1
            elif not true_is_c and not predicted_is_c:
                tn[c] += 1
            else:
                fn[c] += 1

# Missile, Balloon, Castle

print("True Positives: ", tp)
print("False Positives: ", fp)
print("True Negatives: ", tn)
print("False Negatives: ", fn)

True Positives:  [234, 208, 180]
False Positives:  [0, 0, 0]
True Negatives:  [607, 607, 700]
False Negatives:  [116, 142, 77]


In [34]:
tp = np.array(tp)
fp = np.array(fp)
tn = np.array(tn)
fn = np.array(fn)

accuracy = (tp + tn) / (tp + fp + tn + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print('Classes: ', classes)
print("Accuracy: ", accuracy * 100)
print("Precision: ", precision * 100)
print("Recall: ", recall * 100)
print("F1: ", f1 * 100)

print()
print('Mean accuracy: ', accuracy.mean() * 100)
print('Mean precision: ', precision.mean() * 100)
print('Mean recall: ', recall.mean() * 100)
print('Mean F1: ', f1.mean() * 100)

Classes:  ['Missile', 'Balloon', 'Castle']
Accuracy:  [87.87878788 85.16196447 91.95402299]
Precision:  [100. 100. 100.]
Recall:  [66.85714286 59.42857143 70.03891051]
F1:  [80.1369863  74.55197133 82.3798627 ]

Mean accuracy:  88.33159177986764
Mean precision:  100.0
Mean recall:  65.44154159718362
Mean F1:  79.02294010925453
